# Generating the engineer data for TalentMatch

Real staffing/bench data is private company data, so there is no public dataset of engineers
rated on these things. I generate a realistic synthetic bench instead. To make it behave like
real engineers I add several sensible correlations rather than picking every number at random.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from faker import Faker

fake = Faker()
Faker.seed(42)
np.random.seed(42)

N = 3000  # how many engineers to make

## 2. Seniority, experience and domain

Seniority is the core trait. More senior people have more years of experience, and people with
more years tend to know their domain more deeply.

In [2]:
# seniority is roughly a bell curve
seniority = np.clip(np.random.normal(50, 20, N), 0, 99)

# more senior people have more years of experience
years = np.clip(seniority / 99 * 22 + np.random.normal(0, 2, N), 0, 30)

# more years usually means deeper domain knowledge
domain = np.clip(30 + years * 1.2 + np.random.normal(0, 14, N), 0, 99)

## 3. Client communication

Senior people tend to spend more time in front of clients.

In [3]:
communication = np.clip(0.5 * seniority + np.random.normal(25, 14, N), 0, 99)

## 4. Region and timezone overlap

Timezone overlap is measured against a US head office, so it depends on where the engineer is.

In [4]:
regions = np.random.choice(["Americas", "EMEA", "APAC"], N)

# how much each region overlaps with US working hours
region_overlap = {"Americas": 82, "EMEA": 55, "APAC": 28}
timezone_base = np.array([region_overlap[r] for r in regions])
timezone = np.clip(timezone_base + np.random.normal(0, 10, N), 0, 99)

## 5. Bandwidth

Very senior people are often spread across several projects, so they tend to have a bit less
free bandwidth.

In [5]:
bandwidth = np.clip(np.random.normal(65, 20, N) - seniority / 99 * 18, 0, 99)

## 6. Discipline, vertical and names

Discipline is the team the engineer belongs to. Backend and frontend are the most common, the
rest less so.

In [6]:
disciplines = np.random.choice(
    ["Frontend", "Backend", "Data / Database", "DevOps", "Cloud", "Networking", "AI / ML", "Other"],
    N,
    p=[0.18, 0.22, 0.13, 0.12, 0.11, 0.08, 0.10, 0.06],
)

verticals = np.random.choice(
    ["FinTech", "Healthcare", "E-commerce", "SaaS", "Gaming", "Media",
     "Logistics", "EdTech", "Telecom", "Government", "Other"], N)

names = [fake.name() for _ in range(N)]

## 7. Put it together and save

In [7]:
engineers = pd.DataFrame({
    "name": names,
    "discipline": disciplines,
    "region": regions,
    "vertical": verticals,
    "years_experience": np.round(years).astype(int),
    "seniority": np.round(seniority).astype(int),
    "domain": np.round(domain).astype(int),
    "communication": np.round(communication).astype(int),
    "timezone": np.round(timezone).astype(int),
    "bandwidth": np.round(bandwidth).astype(int),
})

engineers.to_csv("engineers.csv", index=False)
print("saved engineers.csv with", len(engineers), "engineers")
engineers.head()

saved engineers.csv with 3000 engineers


,name,discipline,region,vertical,years_experience,seniority,domain,communication,timezone,bandwidth
0,Allison Hill,Frontend,EMEA,Logistics,10,60,26,66,49,80
1,Noah Rhodes,Networking,Americas,Gaming,9,47,32,64,85,31
2,Angie Henderson,Frontend,Americas,SaaS,13,63,33,63,86,98
3,Daniel Wagner,Backend,EMEA,Government,22,80,48,38,49,44
4,Cristian Santos,Other,APAC,Media,11,45,40,45,40,56
